In [1]:
import pandas as pd
url = "https://raw.githubusercontent.com/dylanmalis-collab/Exoplanet_Life/refs/heads/elt_v2/Exoplanet_DBT/data_exports/raw_exoplanets_enriched.csv"
df = pd.read_csv(url)

C:\Users\berna\AppData\Local\Temp\ipykernel_1660\287856858.py:3: DtypeWarning: Columns (5,100,103,242,350,353,359,362,368,396,399,405,408) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(url)


In [2]:
#première selection grossière
df = df[["pl_name", "hostname", "pl_orbsmax_calc","pl_rade", "pl_bmasse_calc", "pl_dens_calc", "pl_orbeccen", "pl_insol_calc", "pl_eqt_calc","st_spectype_group_en", "st_teff", "st_rad", "st_mass", "st_met", "st_lum_lin_calc", "st_dens", "Gravity_G_earth_calc", "masse_10_24kg_calc","pl_type", "sim_earth", "eau_possible","pt_habitable"]]

In [3]:
#transformation (Nan et création des colonne missing)
df["pl_orbeccen"] = df["pl_orbeccen"].fillna(df["pl_orbeccen"].median())

df["st_teff_missing"] = df["st_teff"].isna().astype(int)

# 1. imputation par groupe
df["st_teff"] = df.groupby("st_spectype_group_en")["st_teff"].transform(
    lambda x: x.fillna(x.median())
)

# 2. fallback global
df["st_teff"] = df["st_teff"].fillna(df["st_teff"].median())

df["st_mass_missing"] = df["st_mass"].isna().astype(int)

# 1. imputation par groupe
df["st_mass"] = df.groupby("st_spectype_group_en")["st_mass"].transform(
    lambda x: x.fillna(x.median())
)

# 2. fallback global
df["st_mass"] = df["st_mass"].fillna(df["st_mass"].median())

df["st_rad_missing"] = df["st_rad"].isna().astype(int)

# 1. imputation par groupe
df["st_rad"] = df.groupby("st_spectype_group_en")["st_rad"].transform(
    lambda x: x.fillna(x.median())
)

# 2. fallback global
df["st_rad"] = df["st_rad"].fillna(df["st_rad"].median())

df["st_met_missing"] = df["st_met"].isna().astype(int)

# 1. imputation par groupe
df["st_met"] = df.groupby("st_spectype_group_en")["st_met"].transform(
    lambda x: x.fillna(x.median())
)

# 2. fallback global
df["st_met"] = df["st_met"].fillna(df["st_met"].median())

df["st_dens_missing"] = df["st_dens"].isna().astype(int)

# 1. imputation par groupe
df["st_dens"] = df.groupby("st_spectype_group_en")["st_dens"].transform(
    lambda x: x.fillna(x.median())
)

# 2. fallback global
df["st_dens"] = df["st_dens"].fillna(df["st_dens"].median())

d:\Program Files\Python314\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
d:\Program Files\Python314\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
d:\Program Files\Python314\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
d:\Program Files\Python314\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
d:\Program Files\Python314\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1213: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


In [4]:
# juste pl_insol
df4 = df[["pl_orbsmax_calc","pl_rade", "pl_bmasse_calc", "pl_dens_calc", "pl_orbeccen", "st_teff", "st_rad", "st_mass", "st_met", "st_lum_lin_calc", "st_dens", "sim_earth","pl_insol_calc"]]

In [5]:
from sklearn.model_selection import train_test_split

X = df4
y = df["pt_habitable"]

X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2, stratify=y, random_state=42)

In [7]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.ensemble import RandomForestClassifier
#from sklearn.linear_model import LogisticRegression

cv = RepeatedStratifiedKFold( n_splits=5, n_repeats=3, random_state=42 )

In [ ]:
rf = RandomForestClassifier(random_state=42)

rf_param_grid = {
    "n_estimators": [200, 300, 400, 500, 600],
    "max_depth": [5, 10, 15],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 5],
    "class_weight": ["balanced"],
    "max_features": ["sqrt", "log2", 0.5],
    "criterion": ["gini", "entropy"],
    "min_impurity_decrease": [0.0, 0.001]
}

rf_grid = GridSearchCV(
    rf,
    rf_param_grid,
    cv=cv,
    scoring="average_precision",
    refit=True,
    n_jobs=-1
)

rf_grid.fit(X_train, y_train)

print("\nRANDOM FOREST")
print("Best average_precision:", rf_grid.best_score_)
print("Best params:", rf_grid.best_params_)


RANDOM FOREST
Best F1: 0.9111111111111111
Best params: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 5, 'max_features': 0.5, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300}


In [9]:
model_rf1 = rf_grid.best_estimator_

In [14]:
from sklearn.metrics import average_precision_score

y_proba = model_rf1.predict_proba(X_test)[:, 1]

ap = average_precision_score(y_test, y_proba)

print(" Average Precision:", ap)

 Average Precision: 1.0


In [15]:
from sklearn.metrics import precision_recall_curve, f1_score
import numpy as np

precision, recall, thresholds = precision_recall_curve(y_test, y_proba)

# calcul F1 (avec protection division par zéro)
f1_scores = (2 * precision * recall) / (precision + recall + 1e-9)

# IMPORTANT : on aligne avec thresholds
f1_scores = f1_scores[:-1]   # on enlève le dernier point

best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print("Best threshold:", best_threshold)

y_pred = (y_proba >= best_threshold).astype(int)

print("F1 final:", f1_score(y_test, y_pred))

Best threshold: 0.1466639311252936
F1 final: 1.0


In [16]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(
    model_rf1,
    X_train,
    y_train,
    cv=cv,
    scoring="average_precision",
    n_jobs=-1
)


In [17]:
print("\n📈 CV stability (Average Precision)")
print("Mean:", cv_scores.mean())
print("Std :", cv_scores.std())


📈 CV stability (Average Precision)
Mean: 0.9111111111111111
Std : 0.13076224773266468


In [19]:
import numpy as np
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

threshold = 0.1466639311252936  # ton seuil optimisé

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

f1_scores = []

for train_idx, val_idx in cv.split(X_train, y_train):

    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    model = clone(model_rf1)
    model.fit(X_tr, y_tr)

    y_proba = model.predict_proba(X_val)[:, 1]
    y_pred = (y_proba >= threshold).astype(int)

    f1 = f1_score(y_val, y_pred)
    f1_scores.append(f1)

print("F1 CV mean:", np.mean(f1_scores))
print("F1 CV std :", np.std(f1_scores))

F1 CV mean: 0.7171428571428571
F1 CV std : 0.28290042958826433
